# Knowledge Distillation for Retrieval — Teacher–Student Transfer (MarginMSE)

The reranking sub-track's payoff. The cross-encoder gives the most expressive relevance score
$h([q;d])$, but a joint forward pass per query–document pair forbids precomputation, so it can only
ever be a *reranker*. This notebook **distills** that expensive teacher into a cheap, precomputable
dual-encoder *student* with **MarginMSE** (Hofstätter et al., 2020): match the teacher's per-query
score *margin*, not its absolute level.

This narrative notebook imports the tested reference module
[`retrieval_distillation.py`](retrieval_distillation.py) — which **owns every number** — and walks
the four movements section by section. Run top to bottom; every pedagogical claim is an `assert` in
the module.

In [ ]:
import pathlib, sys
import numpy as np

# Import the canonical module (it owns the numbers); it puts every prerequisite dir on sys.path.
sys.path.insert(0, str(pathlib.Path.cwd()))
import retrieval_distillation as rd

s = rd._setup()
T_clean, T_teacher, truth = s["T_clean"], s["T_teacher"], s["truth"]
print("teacher (cross-encoder) recall@1:", round(rd.topk_recall(T_teacher, truth, 1), 3))
print("teacher score matrix:", T_teacher.shape, "  intrinsic rank:", np.linalg.matrix_rank(T_teacher))

## Movement 1 — MarginMSE and translation-invariance (the theorem)

The all-pairs MarginMSE compares every within-query document *pair*:

$$\mathcal{L}(S,T)=\sum_i\sum_{j,k}\big[(S_{ij}-S_{ik})-(T_{ij}-T_{ik})\big]^2 .$$

Using the variance identity $\sum_{j,k}(a_j-a_k)^2 = 2n_d\sum_j(a_j-\bar a)^2$, this collapses to a
**centered Frobenius distance**

$$\mathcal{L}(S,T)=2\,n_d\,\lVert SC - TC\rVert_F^2,\qquad C = I-\tfrac1{n_d}\mathbf 1\mathbf 1^\top,$$

where $C$ subtracts each query's mean document score. Adding a per-query offset
$T\mapsto T + b\mathbf 1^\top$ leaves $TC$ unchanged ($\mathbf 1^\top C = 0$): **the margin loss is
blind to per-query level.** That translation-invariance is the rigorous hinge of the whole topic.

In [ ]:
rng = np.random.default_rng(0)
S = rng.standard_normal((6, 5)); T = rng.standard_normal((6, 5))
print("all-pairs MarginMSE (brute) :", round(rd.margin_mse_bruteforce(S, T), 6))
print("2 n_d ||SC - TC||^2  (reduction):", round(rd.margin_mse(S, T), 6))

# translation-invariance: add an arbitrary per-query offset; the margin loss does not move.
b = rng.standard_normal(6)
print("margin loss, T          :", round(rd.margin_mse(S, T), 6))
print("margin loss, T + b 1^T  :", round(rd.margin_mse(S, rd.inject_offset(T, b)), 6), "(unchanged)")
print("pointwise loss moves    :", round(float(np.sum((S-T)**2)), 4),
      "->", round(float(np.sum((S-rd.inject_offset(T,b))**2)), 4))

## Movement 2 — the closed-form distilled student (Eckart–Young–Mirsky)

No SGD: the optima are truncated SVDs. The **pointwise**-MSE-optimal rank-$d$ student is
$\operatorname{best\_rank}_d(T)$; the **margin**-optimal rank-$d$ student is
$\operatorname{best\_rank}_d(TC)$ — the per-query-*centered* teacher. Because $TC$ has zero row
sums, so does its truncated SVD, so it is a genuine rank-$d$ dual encoder (realizable as $QG^\top$)
*and* exactly margin-optimal. The embedding-dimension **rank ceiling still binds**: at
$d\ge\operatorname{rank}(TC)$ the student reproduces the teacher's ranking exactly, never better.

In [ ]:
# The constant baseline the teacher carries (its overall score level) is the top singular value;
# centering removes it, so the margin target spends its whole spectrum on RANKING.
print("singular values  T  :", np.round(np.linalg.svd(T_teacher, compute_uv=False), 2))
print("singular values  TC :", np.round(np.linalg.svd(rd.center_rows(T_teacher), compute_uv=False), 2),
      " <- the 42.x baseline is gone")

# the margin student factors into genuine d-dim dual-encoder embeddings Q G^T.
Qs, Gs = rd.realize_student(T_teacher, rd.D_STAGE, margin=True)
ok = np.allclose(rd.score_matrix(Qs, Gs), rd.distill_margin(T_teacher, rd.D_STAGE), atol=1e-9)
print(f"realized rank-{rd.D_STAGE} dual encoder reproduces the student:", ok)

## Movement 3 — margin beats pointwise at restricted rank, and the cost payoff

A cross-encoder's scores are per-query **miscalibrated** (different queries, different score scales) —
the documented reason MarginMSE exists. The pointwise student wastes a slice of its $d$-dimensional
budget reproducing that level; the margin student is blind to it and spends every dimension on
ranking. At the restricted rank $d=$ `D_STAGE` the margin student **out-ranks** the pointwise one,
while the teacher's recall is untouched (an offset preserves every argmax). The payoff: the student
gives cross-encoder-quality ranking at **dual-encoder inference cost** — precompute the document
embeddings once, answer queries by MIPS, never run a per-pair joint forward pass.

In [ ]:
for row in rd.rank_recall_curve(T_teacher, truth, rd.D_GRID):
    star = "  <- D_STAGE" if row["d"] == rd.D_STAGE else ""
    print(f"  d={row['d']}: pointwise {row['pointwise_r1']:.3f}  margin {row['margin_r1']:.3f}"
          f"  (teacher {row['teacher_r1']:.3f}){star}")

corpus = rd.CORPUS_HEADLINE
print(f"\nteacher inference cost : {rd.teacher_inference_cost(corpus):,.0f} / query")
print(f"student inference cost : {rd.student_inference_cost(corpus):,.0f} / query")
print(f"speedup                : {rd.distill_speedup(corpus):.0f}x")

## Movement 4 — dark knowledge, and an honest in-sample finding

The teacher's *graded* scores on the negatives say **which** wrong document is more dangerous —
information a one-hot label lacks (Hinton et al., 2015). We mine the hardest negative per query from
the labeled DPR pool (`mine_nearest`, the negative-sampling prerequisite) and read off the teacher's
margin on those pairs: it is **graded** (a real spread), where the binary margin is the constant $1$.

The honest twist, surfaced by build-and-run: on this clean, block-structured, **in-sample** toy the
binary ground-truth target compresses to low rank *at least as well* as the soft teacher. The classic
soft-beats-hard advantage is a **generalization** phenomenon (scarce held-out labels) that a
closed-form in-sample fit cannot show — a rigorFlag, not a contradiction.

In [ ]:
hm = rd.teacher_hard_margins(T_clean, s["pool"], rd.K_MINE)
print(f"teacher margins on mined hard pairs: n={len(hm)}, mean={hm.mean():.3f}, std={hm.std():.3f}",
      "(graded; binary margin = 1)")
print("\nsoft (teacher) vs hard (binary) recall@1 by rank:")
for row in rd.soft_vs_hard_curve(T_teacher, truth, rd.D_GRID):
    print(f"  d={row['d']}: soft {row['soft_r1']:.3f}   hard {row['hard_r1']:.3f}")

## Verification harness and viz constants

Every claim above is an `assert` in the module; the viz constants below are mirrored to the decimal in
`RetrievalDistillationLaboratory.tsx`.

In [ ]:
rd._run_all()

In [ ]:
rd.viz_constants()